# A2.2 — Les features de métadonnées

Un même fragment ne veut pas dire la même chose en mode positif et en mode négatif, ni selon l'adduct ou l'énergie
de collision. Et la masse neutre renseigne sur la taille de la molécule. `meta_features` encode ces informations en
un petit vecteur de nombres, à placer à côté du spectre biné (A2.1).

Code : `src/casmi/features.py`.
- `fit_meta_encoder(train, adduct_encoding)` apprend les encodages **sur le train seulement** : les adducts et les
  groupes gardés, et l'imputeur de l'énergie de collision ;
- `meta_features(df, encoder)` applique ces encodages à n'importe quel tableau (train, validation, test).
  Il renvoie un DataFrame dont les noms de colonnes sont les noms des features.

**Ce qui entre, ce qui n'entre pas.**

| Colonne | Utilisée ? | Pourquoi |
|---|---|---|
| `ionization_mode`, `adduct`, `precursor_mz`, `instrument_type`, énergie de collision, pics | oui | présentes dans le test |
| `normalized_smiles`, `inchikey14`, `precursor_error_ppm` | **non** | fuites : calculées à partir de la réponse |
| `ingest_lib` | **non** | absente du test |
| `base_peak_intensity` | **non** | proposée exclue (conventions des bibliothèques, aucun imputeur fiable, voir A1.2) |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import numpy as np
import pandas as pd

from casmi.clean import load_clean_meta
from casmi.features import META_INPUT_COLUMNS, MIN_ADDUCT_COUNT, adduct_groups, fit_meta_encoder, meta_features
from casmi.io import load_meta, load_test
from casmi.mass import compute_neutral_masses
from casmi.missing import ce_summary, nce_to_ev
from casmi.split import load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

# Les colonnes d'entrée, plus keep / query_ok (nettoyage) et le split
clean = load_clean_meta()[["row_id", "inchikey14", "keep", "query_ok"]]
df = load_meta(META_INPUT_COLUMNS + ["num_peaks"]).merge(clean, on="row_id")
df = df.merge(load_split(), on="inchikey14", how="left")

train = df[(df["split"] == "train") & df["keep"]]  # pour apprendre : spectres fiables
dev = df[df["dev"] & df["query_ok"]]  # pour évaluer : seulement les filtres sans la structure
test = load_test()
for name, d in {"train": train, "dev": dev, "test": test}.items():
    print(f"{name:6s}: {len(d):>9,} spectres".replace(",", " "))

## 1. L'adduct : deux encodages à comparer

### Premier encodage : le *one-hot*

Une colonne par adduct, qui vaut 1 si le spectre a cet adduct. Mais le train en compte 112, dont beaucoup très rares :
une colonne pour 8 spectres n'apprendrait rien. On garde les adducts assez fréquents, et les autres partagent une
colonne « autre ».

In [ ]:
counts = train["adduct"].value_counts()
rows = []
for threshold in [100_000, 10_000, 1_000, 100, 10]:
    kept = counts.index[counts >= threshold]
    rows.append({"au moins … spectres": threshold, "adducts gardés": len(kept),
                 "% du train couvert": round(counts[kept].sum() / len(train) * 100, 2),
                 "% du test couvert": round(test["adduct"].isin(kept).mean() * 100, 2)})
pd.DataFrame(rows).set_index("au moins … spectres")

**Choix : au moins 1 000 spectres** (`MIN_ADDUCT_COUNT`), soit 16 adducts. Ils couvrent 99,8 % du train et tout le
test visible. Abaisser le seuil ajouterait des colonnes pour 0,2 % des spectres.

Le seuil est calculé sur le train : un adduct du test absent de la liste tombe dans « autre », sans erreur.

### Second encodage : l'adduct décomposé

Le *one-hot* traite chaque adduct comme une classe à part : `[M+Na]+` et `[2M+Na]+` n'ont rien en commun pour le
modèle. La décomposition décrit l'adduct par ce qui le compose :
- `adduct_nM` : le nombre de molécules M (2 pour tous les dimères) ;
- `adduct_charge` : la charge signée (+1, -1, +2…) ;
- `adduct_groupe=…` : le nombre de chaque groupe ajouté (positif) ou perdu (négatif).

Les groupes sont des **blocs chimiques**, pas des atomes : le formiate (CH₂O₂) s'accroche et part en entier, le
décomposer en C, H et O n'aurait pas de sens. Deux règles d'écriture :
- un terme d'un seul élément compte ses atomes : `+Na2` vaut 2 Na, `-H2` vaut -2 H ;
- les autres sont écrits dans l'ordre de Hill (C, H, puis l'ordre alphabétique), pour que `NH4` et `H4N` soient
  le même groupe (`H4N`).

In [ ]:
examples = ["[M+H]+", "[M-H]-", "[2M+Na]+", "[M+CH2O2-H]-", "[M+NH4]+", "[M-2H2O+H]+", "[M+2H]2+", "[2M+Na-2H]-"]
pd.DataFrame([{"adduct": a, "nM": adduct_groups(a)[0], "charge": adduct_groups(a)[1], **adduct_groups(a)[2]}
              for a in examples]).set_index("adduct").fillna(0).astype(int)

In [ ]:
group_counts = {}
for adduct, n in counts.items():
    parsed = adduct_groups(adduct)
    for group in (parsed[2] if parsed else []):
        group_counts[group] = group_counts.get(group, 0) + n
pd.Series(group_counts, name="spectres du train").sort_values(ascending=False).head(12).to_frame()

Avec le même seuil (au moins 1 000 spectres), on garde **8 groupes** : H, Na, CH₂O₂ (formiate), H₄N (ammonium), Cl,
H₂O, K et C₂H₄O₂ (acétate). Les groupes plus rares sont comptés ensemble dans `adduct_groupe=autre` (708 spectres,
0,03 % du train, contre 0,17 % dans la classe « autre » du *one-hot*) ; un adduct non interprétable met
`adduct_inconnu` à 1. Soit 12 colonnes au lieu de 17.

**Lequel est le meilleur ?** On ne le décide pas ici : les deux sont disponibles
(`fit_meta_encoder(train, adduct_encoding="onehot")` ou `"decompose"`) et seront comparés en validation (A4.1),
avec le protocole décrit dans `docs/decisions.md`.

## 2. Les nombres : quelles échelles ?

In [ ]:
mass = compute_neutral_masses(train)
ce = ce_summary(train["collision_energy_ev"])
pd.DataFrame({
    "masse neutre (Da)": pd.Series(mass).describe(),
    "énergie moyenne (eV)": ce["ce_mean"].describe(),
    "nombre de pics": train["num_peaks"].describe(),
}).loc[["min", "25%", "50%", "75%", "max"]].round(1)

Les trois grandeurs ont des échelles très différentes. Un réseau de neurones apprend mal quand une entrée vaut 300 et
une autre 1. On les ramène autour de 0-1 avec des **échelles fixes**, qui ne dépendent pas des données (donc aucun
risque de fuite) :
- masse neutre / 1 000 (médiane 0,34) ;
- énergie / 100 (médiane 0,4) ;
- nombre de pics en **logarithme** (`log10`) : il va de 1 à 56 400, et passer de 10 à 100 pics compte autant que de
  100 à 1 000. Le test n'a pas de colonne `num_peaks` : on compte alors les pics de `ms2_mzs`.

Un adduct non interprétable (comme `[Cat]2+`) n'a pas de masse neutre : on garde alors la masse de l'ion
(`precursor_mz`).

## 3. L'énergie de collision : une liste, et des NA

`collision_energy_ev` est une liste (souvent une seule valeur, parfois plusieurs comme `20,40,60`). On la résume en
trois nombres : minimum, maximum et moyenne. Pour les NA, on applique ce qui a été retenu en A1.2 :
1. la **règle exacte** NCE → eV (`NCE × precursor_mz / 500`) quand le texte d'origine le permet ;
2. sinon, un **gradient boosting** appris sur le train prédit l'énergie moyenne (minimum = maximum = moyenne) ;
3. une **indicatrice** `ce_manquante` dit au modèle que la valeur est imputée.

In [ ]:
na = ce["ce_mean"].isna().to_numpy()
rule = nce_to_ev(train["collision_energy_orig"], train["collision_energy_orig_units"], train["precursor_mz"]).to_numpy()
by_rule = na & ~np.isnan(rule)
print(f"Énergie manquante : {na.sum():,} spectres ({na.mean():.1%})".replace(",", " "))
print(f"  reconstruite par la règle : {by_rule.sum():,}".replace(",", " "))
print(f"  imputée par le modèle     : {(na & ~by_rule).sum():,} ({(na & ~by_rule).mean():.1%})".replace(",", " "))

Pour l'instrument, on suit le plan : une famille d'instrument (6 familles, A1.2) en *one-hot*, et une catégorie
« Inconnu » pour les 1,4 % de NA. Le modèle d'imputation de A1.2 ne retrouve la bonne famille que dans 49 % à 100 %
des cas selon la bibliothèque : une catégorie à part est plus honnête pour si peu de spectres.

## 4. Apprendre sur le train, appliquer partout

In [ ]:
start = time.perf_counter()
encoder = fit_meta_encoder(train)
print(f"Encodeur appris en {time.perf_counter() - start:.0f} s")

start = time.perf_counter()
X_train = meta_features(train, encoder)
print(f"Train encodé en {time.perf_counter() - start:.0f} s : {X_train.shape[1]} features")
X_dev = meta_features(dev, encoder)
X_test = meta_features(test, encoder)
print("Valeurs manquantes :", X_train.isna().sum().sum(), X_dev.isna().sum().sum(), X_test.isna().sum().sum())
print("Mêmes colonnes partout :", X_train.columns.equals(X_dev.columns) and X_train.columns.equals(X_test.columns))

# L'encodage décomposé : seules les colonnes d'adduct changent
encoder_dec = fit_meta_encoder(train, adduct_encoding="decompose")
X_dev_dec = meta_features(dev, encoder_dec)
X_test_dec = meta_features(test, encoder_dec)
print(f"Encodage décomposé : {X_dev_dec.shape[1]} features, groupes gardés : {encoder_dec.groups}")

In [ ]:
pd.DataFrame({"train": X_train.mean(), "dev": X_dev.mean(), "test visible": X_test.mean()}).round(3)

In [ ]:
cols = X_dev_dec.filter(like="adduct").columns
pd.DataFrame({"dev": X_dev_dec[cols].mean(), "test visible": X_test_dec[cols].mean()}).round(3)

Le tableau donne la **moyenne** de chaque feature (pour une colonne *one-hot*, c'est la part des spectres). Il
résume les différences déjà vues en A1.3 :
- le test visible est **100 % timsTOF**, sans énergie manquante, et presque toujours en `[M+H]+` ou `[M-H]-` ;
- ses spectres ont plus de pics (`log_num_peaks` 2,3 contre 1,7, soit environ 200 pics contre 50) ;
- aucun adduct dimère (`[2M+…]`) dans le test, alors qu'ils font environ 14 % du train.

`dev` ressemble au train : c'est attendu, il est tiré au hasard parmi les molécules.

## 5. Pour la suite (A2.3)

Le vecteur complet d'un spectre = le spectre biné (10 000 cases, creux) + ces features (31 en *one-hot*, 26 en
décomposé, denses). On les colle
avec `scipy.sparse.hstack`. L'encodeur doit être appris une seule fois sur le train, puis réutilisé tel quel pour la
validation et le test.